# DataPath — Week 3: Data Cleaning — SOLUTIONS
**Art.Ificial · Spring 2026**

Today we're doing two things:
1. Learning the basic pandas commands you'll use all project
2. Cleaning our dataset so it's ready for analysis

Work through each cell top to bottom. Press **Shift + Enter** to run a cell.

---
## Part 1 — Pandas Basics
Pandas is the main Python library for working with data. Think of it as Excel inside Python.

A **DataFrame** is just a table — rows and columns, like a spreadsheet.

In [ ]:
# First, import the libraries we need
# pandas = working with data tables
# numpy = math operations

import pandas as pd
import numpy as np

print("Libraries loaded successfully!")

In [ ]:
# Load the dataset
# Make sure the CSV file is in the same folder as this notebook
# or provide the full path

df = pd.read_csv("../data/raw/YouTube-TikTok Trends Dataset/youtube_shorts_tiktok_trends_2025.csv")

print(f"Dataset loaded! Shape: {df.shape}")
print(f"That means {df.shape[0]} rows and {df.shape[1]} columns")

In [ ]:
# .head() shows the first 5 rows
# This is always the first thing you run on a new dataset
# Just to see what you're working with

df.head()

In [ ]:
# .info() shows every column, its data type, and how many non-null values it has
# 'non-null' means the cell actually has a value (not empty)
# If a column has fewer non-null values than total rows, it has missing data

df.info()

In [ ]:
# .isnull().sum() counts how many missing values are in each column
# Any column with a number > 0 has gaps we need to deal with

print("Missing values per column:")
print(df.isnull().sum())

In [ ]:
# .describe() gives you basic statistics for every numeric column
# min, max, mean, median (50%) — useful for spotting weird values

df.describe()

In [ ]:
# Let's see what platforms are in the dataset
# .value_counts() counts how many times each unique value appears

print("Platform breakdown:")
print(df["platform"].value_counts())

print("\nTrend labels:")
print(df["trend_label"].value_counts())

In [ ]:
# Let's look at the columns we actually care about for this project
# You can select specific columns like this:

df[["platform", "views", "likes", "duration_sec", "engagement_rate"]].head(10)

---
## Part 2 — Data Cleaning

Cleaning means:
- Removing duplicate rows
- Dropping rows with missing values in columns we need
- Removing rows with nonsense values (e.g. 0 views)
- Adding useful derived columns
- Keeping only the columns we actually need

We always work on a **copy** of the original data so we never lose anything.

In [ ]:
# Step 0 — make a copy so we never touch the original
# Think of df as the locked original, clean as our working copy

clean = df.copy()
print(f"Starting rows: {len(clean)}")

In [ ]:
# Step 1 — remove duplicate rows
# A duplicate is a row that is identical to another row in every column
# These can sneak in when datasets are merged or scraped multiple times

before = len(clean)
clean = clean.drop_duplicates()
after = len(clean)

print(f"Removed {before - after} duplicate rows")
print(f"Rows remaining: {after}")

In [ ]:
# Step 2 — drop rows with missing values in critical columns
# We can't analyze a video if we don't know its views, duration, or platform
# .dropna(subset=[...]) removes any row where ANY of those columns is empty

critical_columns = ["views", "likes", "duration_sec", "platform", "engagement_rate"]

before = len(clean)
clean = clean.dropna(subset=critical_columns)
after = len(clean)

print(f"Removed {before - after} rows with missing critical values")
print(f"Rows remaining: {after}")

In [ ]:
# Step 3 — remove nonsense values
# A video with 0 views or 0 duration shouldn't exist
# Shorts are max 3 minutes (180 seconds) — anything longer isn't a Short

# In Part 1 you saw how boolean filtering works: df[df["column"] > value]
# Now you write them. The first one is done as an example.

before = len(clean)

clean = clean[clean["views"] > 0]           # keep rows where views > 0
clean = clean[clean["likes"] >= 0]          # SOLUTION: likes can be 0, not negative
clean = clean[clean["duration_sec"] > 0]    # SOLUTION: duration must be positive
clean = clean[clean["duration_sec"] <= 180] # SOLUTION: max short length is 180s

after = len(clean)
print(f"Removed {before - after} rows with nonsense values")
print(f"Rows remaining: {after}")

In [ ]:
# Step 4 — handle outliers
# Outliers are extreme values that can skew your analysis
# We'll cap views at the 99th percentile
# This means: find the value that 99% of videos are below, remove anything above it

p99 = clean["views"].quantile(0.99)
print(f"99th percentile of views: {p99:,.0f}")
print(f"Max views before: {clean['views'].max():,.0f}")

before = len(clean)
clean = clean[clean["views"] <= p99]
after = len(clean)

print(f"\nRemoved {before - after} extreme outlier rows")
print(f"Max views after: {clean['views'].max():,.0f}")
print(f"Rows remaining: {after}")

In [ ]:
# Step 5 — add derived columns
# These are new columns we calculate from existing ones
# They will be useful features for our model later

# title_length: how many characters is the title?
clean["title_length"] = clean["title"].str.len()

# duration bucket: group videos into length categories
# This helps us answer: do shorter videos perform better?
def duration_bucket(seconds):
    if seconds <= 15:
        return "0-15s"
    elif seconds <= 30:
        return "15-30s"
    elif seconds <= 60:
        return "30-60s"
    else:
        return "60-180s"

clean["duration_bucket"] = clean["duration_sec"].apply(duration_bucket)

# is_viral: binary column based on our agreed definition
# Top 10% of views = viral
viral_threshold = clean["views"].quantile(0.90)
clean["is_viral"] = (clean["views"] >= viral_threshold).astype(int)

print(f"Viral threshold (top 10%): {viral_threshold:,.0f} views")
print(f"Viral videos: {clean['is_viral'].sum():,}")
print(f"Non-viral videos: {(clean['is_viral'] == 0).sum():,}")
print(f"\nDuration bucket breakdown:")
print(clean["duration_bucket"].value_counts())

In [ ]:
# Step 6 — keep only the columns we need
# Our dataset has 57 columns — we only need about 20
# Keeping only what we need makes everything faster and easier to read

# YOUR TURN: select only the columns below from clean
# Hint: in Part 1 you did df[["col1", "col2"]] to pick specific columns — same idea here

keep_columns = [
    # identity
    "platform", "category", "country",
    # performance metrics
    "views", "likes", "comments", "shares", "saves", "engagement_rate", "completion_rate",
    # video features
    "duration_sec", "duration_bucket", "upload_hour", "publish_dayofweek",
    "is_weekend", "music_track", "sound_type", "has_emoji",
    # derived
    "title_length", "is_viral", "trend_label", "creator_tier",
]

clean = clean[keep_columns]  # SOLUTION: pass the list inside df[[...]] to select columns

print(f"Columns reduced from {len(df.columns)} to {len(clean.columns)}")
print(f"Final shape: {clean.shape}")
clean.head()

In [ ]:
# Step 7 — final check before saving
# YOUR TURN: use what you learned in Part 1 to answer these three questions.
# Each one has a hint pointing back to the Part 1 cell where you saw it.

print("=== FINAL DATASET SUMMARY ===")
print(f"Total rows: {len(clean):,}")
print(f"Total columns: {len(clean.columns)}")

# Q1: How many missing values are in each column?
print(f"\nMissing values:")
print(clean.isnull().sum())  # SOLUTION

# Q2: What's the platform breakdown (how many videos per platform)?
print(f"\nPlatform split:")
print(clean["platform"].value_counts())  # SOLUTION

# Q3: How many viral vs non-viral videos are there?
print(f"\nViral vs non-viral:")
print(clean["is_viral"].value_counts())  # SOLUTION

# This one is filled in for you — just run it
print(f"\nViews stats:")
print(clean["views"].describe().apply(lambda x: f"{x:,.0f}"))

In [ ]:
# Step 8 — save the clean dataset
# index=False means don't write the row numbers as a column

clean.to_csv("clean_dataset.csv", index=False)

print("Saved to clean_dataset.csv")
print("Upload this file to Google Drive in the /data/clean folder")
print("Push this notebook to GitHub in the /notebooks folder")

---
## Part 3 — Quick Sanity Check
Load the saved CSV back and confirm it looks exactly right.
This is good practice — always verify your saved file loads correctly.

In [ ]:
# Load the clean file back to confirm it saved correctly
verify = pd.read_csv("clean_dataset.csv")

print(f"Rows: {len(verify):,}")
print(f"Columns: {list(verify.columns)}")
verify.head()

---
## Done!

**What we built today:**
- A clean dataset with ~48,000 videos from TikTok and YouTube
- Removed duplicates, nulls, and nonsense values
- Added `is_viral`, `duration_bucket`, and `title_length` columns
- Reduced from 57 columns to 22 focused columns

**Before you leave:**
1. Upload `clean_dataset.csv` to Google Drive `/data/clean`
2. Push this notebook to GitHub `/notebooks`

**Next week (Week 4):** We use this clean data to start finding patterns — do shorter videos go more viral? Does upload time matter? Does platform matter?